# 01.3 — Functions and lambdas

How to declare, call, and pass functions in Python. The biggest mindset shifts vs MATLAB:

* Functions live anywhere — same file as the caller, separate module, inside another function.
* Default arguments and keyword arguments are first-class language features (no `CheckVararginPairs` boilerplate).
* Functions are values you can pass around, store in dicts, and return from other functions.

**Prerequisite:** [01.1 syntax basics](01.1_syntax_basics.ipynb).

## Section 1 — What MATLAB does

In MATLAB, a function lives in its own `.m` file:

```matlab
% File: my_add.m
function z = my_add(x, y)
% MY_ADD Add two numbers.
z = x + y;
end
```

Optional arguments require the `varargin` cell array and lots of boilerplate (the project uses `CheckVararginPairs` everywhere).

You can also have anonymous (lambda-style) functions:

```matlab
sq = @(x) x.^2;
sq(5)   % 25
```

Python compresses all of this into much less ceremony.

## Section 2 — The Python concepts you need

### 2.1 — Defining and calling functions

`def` opens a function. The body is indented. The (optional) `return` statement provides the result. A triple-quoted string immediately under the `def` line is the **docstring** — Python's equivalent of MATLAB's `%FUNCTION_NAME` header comment, accessible via `help()`.

In [ ]:
def my_add(x, y):
    """Add two numbers."""
    return x + y

print(my_add(2, 3))
help(my_add)

**Differences worth pinning:**

| MATLAB | Python |
|---|---|
| `function z = my_add(x, y)` | `def my_add(x, y):` |
| separate `.m` file required | function lives in any `.py`, defined anywhere |
| `z = x + y;` then end-of-function | `return x + y` |
| `% MY_ADD ...` comment line | triple-quoted docstring as the first statement |

A Python function can have **zero or many `return` statements**. With no explicit `return`, the function returns `None` (Python's `null`).

### 2.2 — Default arguments and keyword arguments

This is where Python is dramatically cleaner than MATLAB.

In [ ]:
def build_model(name, hidden_size=250, dropout=0.5, variational=True):
    """Return a config dict for a model."""
    return {
        "name": name,
        "hidden_size": hidden_size,
        "dropout": dropout,
        "variational": variational,
    }

# Call with positional args only
print(build_model("GRU"))

# Override one parameter by name
print(build_model("LSTM", dropout=0.25))

# Mix positional + keyword
print(build_model("Feedforward", 500, variational=False))

**Keyword arguments** are how the codebase passes long configs. They make calls self-documenting:

```python
ds = MatFileTrialDataset(
    data_dir=cfg.data_dir,
    target_dir=cfg.target_dir,
    data_width=100,
    window_stride=50,
    target_type="Dimension",
)
```

The codebase goes one step further: `MatFileTrialDataset.__init__` begins its parameter list with a bare `*`, which makes every argument **keyword-only** — a positional call like

```python
ds = MatFileTrialDataset(cfg.data_dir, cfg.target_dir, 100, 50, "Dimension")   # TypeError!
```

doesn't even run. For long config-like signatures that's a deliberate design choice: nobody can silently swap two arguments of the same type.

### 2.3 — `*args` and `**kwargs`

When you want a function to accept an arbitrary number of arguments, MATLAB uses `varargin`. Python uses `*args` (positional) and `**kwargs` (keyword).

In [ ]:
def sum_all(*args):
    """Accept any number of positional arguments."""
    return sum(args)

print(sum_all(1, 2, 3))
print(sum_all(*[10, 20, 30]))   # unpack a list into positional args

In [ ]:
def build_with_extras(name, **kwargs):
    """Accept any number of keyword arguments."""
    cfg = {"name": name}
    cfg.update(kwargs)
    return cfg

print(build_with_extras("GRU", lr=1e-3, batch=32))

# Unpack a dict into keyword args with **
extras = {"lr": 1e-3, "batch": 32, "dropout": 0.5}
print(build_with_extras("LSTM", **extras))

**Conventions:**

- `*args` collects positional arguments into a tuple.
- `**kwargs` collects keyword arguments into a dict.
- The names `args` and `kwargs` are convention, not law — but every Python programmer reads them at a glance.
- Inside a CALL, `*` and `**` UNPACK an existing list/dict into individual arguments.

In our codebase: `_build_real_data_split` in `cli.py` uses `**common_kwargs` to forward a dict to `MatFileTrialDataset` without naming each field twice.

### 2.4 — Lambdas (anonymous functions)

Python's lambda is the equivalent of MATLAB's `@(x) ...` syntax. **Lambdas are intentionally limited** — a single expression, no statements — so they stay readable inline.

For anything beyond a one-line expression, use a named `def`.

In [ ]:
# A lambda lives where you'd write a function expression
square = lambda x: x * x
print(square(5))

# More commonly: pass a lambda to a higher-order function.
nums = [1, 2, 3, 4, 5]
squared = list(map(lambda x: x * x, nums))
print(squared)
# Idiomatic alternative: a list comprehension. Most Pythonistas prefer this:
print([x * x for x in nums])

# Sorting by a key
pairs = [("GRU", 250), ("LSTM", 100), ("Feedforward", 1000)]
print(sorted(pairs, key=lambda pair: pair[1]))

**When to use a lambda vs a named function:**

- Lambda: one-liner expression used once, in-place (e.g. as a `key=` argument).
- `def`: any logic with multiple statements, control flow, or that you'll call twice.

PEP 8 explicitly discourages assigning a lambda to a name — `square = lambda x: x*x` is worse than `def square(x): return x * x` because the latter has a name visible in error tracebacks. Use lambdas inline, not as named assignments.

### 2.5 — Functions are values

Anything you can do with an integer or a string, you can do with a function: assign it to a variable, store it in a list/dict, pass it as an argument, return it from another function. This is what makes the **registry pattern** (used heavily in this codebase) feel natural.

In [ ]:
# Store functions in a dict — the registry pattern
def relu(x): return max(0, x)
def square(x): return x * x
def negate(x): return -x

OPS = {
    "relu": relu,
    "square": square,
    "negate": negate,
}

# Look up by name, then call
chosen = OPS["square"]
print(chosen(7))

This is the spirit of `src/neural_data_decoding/models/registry.py` — encoder and classifier **builder functions** are looked up by their string name; each returns a constructed model when called.

Lookups by string are the Python equivalent of MATLAB's `switch` over `ModelName`, but with the cases pluggable from outside the function — far easier to extend.

## Section 3 — The `neural_data_decoding` implementation

The registry's lookup functions in action — string-keyed dispatch over registered builders:

In [ ]:
# Look at register_classifier + build_classifier in the model registry
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/registry.py").read_text().splitlines()
start = next(i for i, line in enumerate(src) if line.startswith("def register_classifier"))
for i in range(start, min(start + 40, len(src))):
    print(f"{i + 1:3} | {src[i]}")

Look for:
- `register_classifier` RETURNS a decorator — classifier builders elsewhere in the codebase register themselves with `@register_classifier("...")`, which stores the function in a module-level dict keyed by string name. (Decorators are functions that wrap other functions — 01.4 touches on them.)
- `def build_classifier(name: str, cfg: ConfigLike)` — the lookup takes the string name plus one `cfg` mapping; each builder reads whatever keys it needs from that mapping (rather than declaring dozens of parameters).
- Type hints (`name: str`, the `ConfigLike` alias) — covered in 01.7.

## Section 4 — Hands-on exercises

### Exercise 4.1 — port a MATLAB function

Port this:

```matlab
function out = scale_value(x, factor, offset)
if nargin < 3
    offset = 0;
end
if nargin < 2
    factor = 1;
end
out = x * factor + offset;
end
```

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
def scale_value(x, factor=1, offset=0):
    return x * factor + offset

print(scale_value(5))                   # 5
print(scale_value(5, factor=2))          # 10
print(scale_value(5, factor=2, offset=1))  # 11

### Exercise 4.2 — use a lambda

Sort the list `[("a", 30), ("b", 10), ("c", 20)]` by the second element, descending.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
pairs = [("a", 30), ("b", 10), ("c", 20)]
print(sorted(pairs, key=lambda p: p[1], reverse=True))

## Section 5 — Common errors

### `TypeError: my_func() missing 1 required positional argument: 'x'`

You called `my_func()` without providing `x`. Either pass it explicitly or give it a default value in the `def`.

### `TypeError: my_func() got an unexpected keyword argument 'foo'`

You called `my_func(foo=...)` but `my_func`'s signature doesn't declare `foo`. Either fix the call or add `**kwargs` to the signature to accept arbitrary keyword args.

### Forgetting to `return`

```python
def add(x, y):
    x + y      # computes the sum, then throws it away — no return!

z = add(1, 2)  # z is None
```

Python doesn't auto-return the last expression like MATLAB. You have to write `return`.

### Mutable default argument bug

Covered in [00.3 the mental model](../00_orientation/00.3_the_matlab_to_python_mental_model.ipynb). Use `None` and create inside the body:

```python
def add_default(item, into=None):
    if into is None:
        into = []
    into.append(item)
    return into
```

### `SyntaxError: invalid syntax` on a multi-line lambda

Lambdas can only be a single expression. If you need multiple statements, use `def`.

## Section 6 — Further reading

- [Python tutorial — Defining functions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions). The official walkthrough.
- [PEP 3102 — keyword-only arguments](https://peps.python.org/pep-3102/). The `*` before keyword-only args (you'll see this in our codebase).
- [PEP 8 — lambda style](https://peps.python.org/pep-0008/#programming-recommendations) — why you shouldn't name a lambda.

Next up: **[01.4 classes and OOP](01.4_classes_and_oop.ipynb)**.